In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand, when

spark = SparkSession.builder \
    .appName("BigDataExperiment") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 17:02:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.1


In [2]:
df = spark.range(
    start=0,
    end=1000000,
    numPartitions=8
)

df = df.withColumn(
    "importe",
    (rand() * 1000).cast("double")
).withColumn(
    "categoria",
    when(col("id") % 3 == 0, "A")
    .when(col("id") % 3 == 1, "B")
    .otherwise("C")
)

print("Particiones:", df.rdd.getNumPartitions())

Particiones: 8


In [3]:
df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/opt/spark-data/bigdata_test")

In [5]:
import time

inicio = time.time()

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/opt/spark-data/bigdata_test")

resultado = df.groupBy("categoria") \
    .sum("importe")

resultado.show()

fin = time.time()

print(f"Tiempo de procesamiento: {fin-inicio:.2f} segundos")

+---------+--------------------+
|categoria|        sum(importe)|
+---------+--------------------+
|        C| 1.665754764744686E8|
|        B|1.6672463776894298E8|
|        A|1.6671370141445136E8|
+---------+--------------------+

Tiempo de procesamiento: 2.01 segundos
